# Intro to KV Caching with Valkey and LMCache

> Store and reload LLM KV-cache tensors in Valkey using LMCache's connector — on CPU, no GPU required.

**Beginner** · Python · ~15 min

**Who is this for:** Developers who want to self-deploy LLMs and optimize inference throughput and latency. This cookbook explains what KV caching is and shows how to set up KV caching with Valkey and [LMCache](https://github.com/LMCache/LMCache), so you can see a cache hit end-to-end on your laptop.

## Why KV caching?

When an LLM processes a prompt, it computes attention **key/value (KV) tensors** for every token. If the same prompt (or a shared prefix, like a system prompt or few-shot examples) shows up again, recomputing those tensors wastes compute and adds latency to the first token. **KV caching stores those tensors and reloads them on the next request instead of recomputing**, which cuts time-to-first-token and frees the accelerator for new work.

[LMCache](https://github.com/LMCache/LMCache) is a KV-cache layer for inference engines (vLLM, SGLang). It offloads KV tensors to external storage and reloads them on a prefix match. **Valkey serves as that external (L2) store**: network-attached and shared across instances. In this demo the cache lives in the Valkey container, so it survives stopping and starting the container but is lost if the container is removed (`docker compose down`). Mount a data volume and enable append-only persistence to keep it across container recreation.

## What you'll do here

You'll drive the same connector class (`ValkeyConnector`) that vLLM uses in production, directly on CPU: store a KV-cache tensor into Valkey, confirm the cache hit, and load it back byte-for-byte. Because this is the production connector rather than a simplified stand-in, the store-and-load path you exercise here is the same one used in production. You're just running it without the GPU inference engine on top. GPU memory is scarce and evicts cached tensors quickly, so LMCache moves them to a larger tier like Valkey that several inference workers can share. Running the full vLLM inference server on top of this connector is the production step. It needs a Linux/GPU host and is covered in [03 - Scaling to Production](03-scaling-to-production.ipynb).

## Prerequisites

- Docker (to run Valkey locally)
- Python 3.10 or newer (LMCache requires `>=3.10,<3.14`)
- No API keys, GPU, or paid services needed

Install the Python dependencies (from this cookbook directory):

```bash
pip install -r requirements.txt
```

This cookbook reaches Valkey in two ways, and both are already pinned in
`requirements.txt`:

- **LMCache's connector** does the KV-cache reads and writes. It uses the
  `valkey-glide-sync` client, which builds on `valkey-glide` — so both packages
  are required (installing only one fails at import).
- **`valkey-cli`** (run via `docker exec`) is used only for quick sanity checks
  like `ping` and `dbsize`.


## Step 1: Start Valkey

```bash
docker compose up -d --wait
```

This starts Valkey bound to `127.0.0.1:6379` (see `docker-compose.yml`).

> ⚠️ **Security:** These examples use no authentication or TLS for simplicity.
> For any non-localhost deployment, enable authentication and TLS.
> See the [Valkey security documentation](https://valkey.io/topics/security/).

In [ ]:
import subprocess

# Confirm Valkey is reachable before we start.
print(subprocess.run(["docker", "exec", "valkey", "valkey-cli", "ping"],
                     capture_output=True, text=True).stdout.strip())

## Step 2: Write and load an LMCache config

LMCache reads its settings from a YAML file (the `LMCACHE_CONFIG_FILE`). The
sample `lmcache_config.yaml` points LMCache at Valkey as the remote backend.
We load it through LMCache's own config parser, so an invalid setting fails now, on your laptop, instead of later on a GPU host.

In [ ]:
from common import load_lmcache_config

config = load_lmcache_config("lmcache_config.yaml")
print("chunk_size:", config.chunk_size)
print("remote_url:", config.remote_url)      # valkey://localhost:6379
print("extra_config:", config.extra_config)

## Step 3: Build LMCache's Valkey key

LMCache keys every stored KV chunk with `lmcache.utils.CacheEngineKey`. The
wire key encodes four things — the model name, the worker id, the chunk hash,
and the **dtype** (the numeric type of the KV-cache tensor values, e.g.
`bfloat16` or `float16`). All four have to match for LMCache to reuse a stored chunk (a cache hit).
KV tensors written as `bfloat16` can't be reused by a worker expecting
`float16`, so the dtype is part of the key. Building one shows exactly what
lands in Valkey.

In [ ]:
from common import build_cache_key

# chunk_hash identifies the token chunk; dtype is fixed to the demo's bfloat16.
key = build_cache_key(chunk_hash=0x75BCD15)
print(key.to_string())   # facebook/opt-125m@1@0@75bcd15@bfloat16
#                          model            │ │ │        └ dtype
#                                    world ─┘ │ └ chunk hash (hex)
#                                    worker ──┘

## Step 4: Connect LMCache's Valkey connector

`ValkeyConnector` is the component LMCache uses to read and write KV chunks in
Valkey. The `valkey://` URL scheme is dispatched to it by LMCache's
`valkey_adapter.py` (`ValkeyConnectorAdapter`). Here we construct it directly,
on CPU, so we can exercise it without a GPU or vLLM.

`common.make_backend_and_connector()` builds a `LocalCPUBackend` plus the
`ValkeyConnector` for our demo model geometry.

In [ ]:
from common import make_backend_and_connector, make_kv_tensor

cpu_backend, connector = await make_backend_and_connector()
print("connector chunk shape:", tuple(connector.meta_shapes[0]),
      "=", connector.full_chunk_size_bytes, "bytes")

## Step 5: Store a KV chunk and watch the cache hit

We allocate a KV-cache tensor (sized to the connector's fixed chunk shape),
`put` it into Valkey under our key, then confirm it's there. In a live
deployment this tensor comes from the model's attention layers; here we fill it
with deterministic data so we can verify the round trip.

In [ ]:
mem, original = make_kv_tensor(cpu_backend, connector)

await connector.put(key, mem)
hit = await connector.exists(key)
print("cache hit in Valkey:", hit)

In [ ]:
# Confirm from Valkey's side that the key is really there.
import subprocess
print(subprocess.run(["docker", "exec", "valkey", "valkey-cli", "dbsize"],
                     capture_output=True, text=True).stdout.strip())

## Step 6: Load the KV chunk back from Valkey

On a real request with a matching prefix, LMCache loads the KV tensor from
Valkey instead of recomputing it. We do the same `get` here and confirm the
bytes are identical to what we stored.

In [ ]:
import torch

got = await connector.get(key)
identical = got is not None and torch.equal(
    got.tensor.to(torch.float32), original.to(torch.float32)
)
print("loaded from Valkey, byte-identical:", identical)

In [ ]:
# Clean up the connector (closes the GLIDE client and worker pool).
await connector.close()

## How it works

| Component | Role |
|-----------|------|
| `LMCacheEngineConfig` | Parsed from YAML by LMCache's loader; tells LMCache which backend and settings to use |
| `CacheEngineKey` | Builds the Valkey key for a KV chunk from model, worker, chunk hash, and dtype |
| `ValkeyConnector` | LMCache's connector — `put`/`get`/`exists` KV chunks in Valkey (the class vLLM drives) |
| `valkey_adapter.py` | Dispatches the `valkey://` URL scheme to `ValkeyConnector` |
| Valkey (L2) | Stores the serialized KV chunk bytes; network-attached, shared, and durable when backed by a volume |

## Where this goes next

- **[02 - Sharing KV Caches Across Instances](02-sharing-kv-caches-across-instances.ipynb):** two workers sharing one Valkey-backed cache.
- **[03 - Scaling to Production](03-scaling-to-production.ipynb):** cluster mode, TLS, monitoring, and wiring this connector under a vLLM server.

## Teardown

```bash
docker compose down
```

---

[← README](README.md) | [02 - Sharing KV Caches Across Instances →](02-sharing-kv-caches-across-instances.ipynb)